# Full Factorial: Illustration of the scaling law of recurrent networks.

The aim of this notebook is to illustrate the effect on training process of hyperparameters changes.

## The Hyperparameters

The hyperparameters that we will consider are:

- Updates: the number of optimizer updates the model will take. *Epochs = Updates * Batch size / Number of sequences*
- Sequence length: axes 1 of the input data, the length of the input sequences. (characters length)
- Number of sequences: axes 0 of the input data, i.e. the number of sequences in the dataset.


### Static Hyperparameters

The following hyperparameters will remain constant throughout the notebook:
- Batch size
- Learning rate (Cosine scheduler applied)
- Token dimension
- Split ratio (train/test)

## Experiment

The analysis are performed through full factorial technique.

## The Training Problem

Given a language grammar, the LSTM will be able to classify sequences of characters as valid or invalid according to the grammar rules.

BNF Definition:

$$
\begin{array}{rcl}
\langle\mathit{string}\rangle   & \mathrel{::=} & \langle\mathit{term}\rangle \\
                              & \mid          & \langle\mathit{string}\rangle \mathbin{\texttt{+}} \langle\mathit{term}\rangle \\[2pt]
\langle\mathit{term}\rangle   & \mathrel{::=} & AB, ED, OK \\[2pt]
\end{array}
$$

In [1]:
"""Static Hyperparameters Configuration"""

BATCH_SIZE = 16
SPLIT_RATIO = 0.9
MAX_LR, MIN_LR = 1e-2, 1e-4

In [2]:
"""Model Architecture"""

from thorcino.activations import Sigmoid
from thorcino.layers.linear import Linear
from thorcino.layers.lstm import LSTM
from thorcino.layers.sequential import Sequential
from thorcino.losses import BinaryCrossEntropyLoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineSchedule
from thorcino.training.trainer import Trainer

def get_trainer(epochs: int):
    model = Sequential(
        LSTM(
            in_feature=3,
            hidden_units=3,
            out_type='n_to_1',
        ),
        Linear(
            in_feature=3,
            out_feature=1,
        ),
        Sigmoid()
    )
    loss = BinaryCrossEntropyLoss()
    optimizer = SGD(model.parameters, MAX_LR)
    scheduler = CosineSchedule(MAX_LR, MIN_LR, epochs)
    trainer = Trainer(
        model,
        loss,
        optimizer,
        scheduler,
    )

    return trainer

## Validation Set

A validation set is used to evaluate the model performance in order to choose the best hyperparameters configuration.
The validation set is shared across all runs.

Considering that the problem is length independent, several sub validation set are build, each set has a different sequence length, this last, increase by a factor of 2, starting from the minimum sequence length used in the experiments.

In [ ]:
"""Create the Validation Set"""

from examples.helpers.dataset import get_dataset

VAL_NUMBER_OF_SEQUENCE = 500
VAL_SEQUENCE_LENGTHS = [16, 32, 64, 128, 256, 512]

val_dataset = []
for seq_len in VAL_SEQUENCE_LENGTHS:
    val_dataset.append(get_dataset(VAL_NUMBER_OF_SEQUENCE, seq_len))

In [ ]:
"""Evaluate Validation Set"""

import numpy as np

from thorcino.dataset.dataset import DataLoader, TensorDataset

def evaluate(trainer: Trainer, val_dataset: list[tuple[np.array, np.array]]) -> dict:
    loss_history, accuracy_history = [], []
    for dataset in val_dataset:
        X, Y = val_dataset
        dl = DataLoader(TensorDataset(X), TensorDataset(Y))

        loss, acc = trainer.eval(dl)

        loss_history.append(loss)
        accuracy_history.append(acc)

    return {
        'loss_history': loss_history,
        'accuracy_history': accuracy_history
    }

## The Training Process

Different trials will be performed for each hyperparameter, increasing its value by a 40% factor each time, every trails metrics will be plotted and compared to the baseline configuration, showing how the hyperparameter affects the training process and model performance.

In [ ]:

from examples.helpers.dataset import get_dataset, preprocess

def run_experiment(epochs: int, eval_step: int, n_sequence: int, sequence_length: int) -> Trainer:
    X, Y = get_dataset(n_sequence, sequence_length)
    train_dl, test_dl = preprocess(X, Y, BATCH_SIZE, SPLIT_RATIO)

    trainer = get_trainer(epochs)

    for e in range(epochs):
        _ = trainer.train_epoch(train_dl)
        
        if e%eval_step == 0:
            _ = trainer.eval(test_dl)

    return trainer

In [ ]:
"""Creating Factors"""

import itertools
from os import listdir, path
import time

hyperparams = {
    'updates': [1200, 2400, 3200],
    'number_of_sequence': [200, 400, 800],
    'sequence_length': [32, 64, 128]
}

## To be a full factorial every factor must has same space size.
first_len = None
for key in hyperparams.keys():
    if first_len == None:
        first_len = len(hyperparams[key])
    assert first_len == len(hyperparams[key]) 

G = itertools.product(range(first_len), repeat=3)


## BACKUP_FOLDER and METRICS_FOLDER must contains same numbers of element and must be sames runs
BACKUP_FOLDER = "./checkpoint/backup"
TRAINING_FOLDER = "./checkpoint/metrics/training"
VALIDATION_FOLDER = "./checkpoint/metrics/validation"

previous_backups, previous_training, previous_validation = [], [], []

if path.isdir(BACKUP_FOLDER): previous_backups =  listdir(BACKUP_FOLDER)
if path.isdir(TRAINING_FOLDER): previous_training = listdir(TRAINING_FOLDER)
if path.isdir(VALIDATION_FOLDER): previous_validation = listdir(VALIDATION_FOLDER)

if len(previous_backups) > 0 and len(previous_training) > 0 and len(previous_validation): 
    assert len(previous_backups) == len(previous_training)
    assert len(previous_backups) == len(previous_validation)

idx_restore = len(previous_backups)
for i, exp in enumerate(list(G)[idx_restore:]):
    idx_n_seq, idx_s_len, idx_updates = exp
    updates, n_seq, s_len = hyperparams['updates'][idx_updates], hyperparams['number_of_sequence'][idx_n_seq], hyperparams['sequence_length'][idx_s_len]

    epochs = updates*BATCH_SIZE / n_seq
    eval_step = int(epochs/10)

    print('-----------------NEW EXPERIMENT STARTED-----------------')
    print(f'experiment hyperparameters: EPOCHS={epochs} UPDATES={updates}, NUMBER_OF_SEQUENCE={n_seq}, SEQUENCE_LENGTH={s_len}')

    now = time.perf_counter()
    trainer = run_experiment(epochs, eval_step, n_seq, s_len)
    experiment_age = int(time.perf_counter() - now)

    artifact_name = f"E{idx_restore+i}__{epochs}_{updates}_{n_seq}_{s_len}__{experiment_age}s.pkl"
    trainer.save(f'{BACKUP_FOLDER}/{artifact_name}')
    trainer.save_metrics(f'{TRAINING_FOLDER}/{artifact_name}')

    evaluation_history = evaluate(trainer, val_dataset)
    trainer._save_artifact(VALIDATION_FOLDER, evaluation_history)

    print(f'experiment age: {experiment_age} seconds')
    print('-------------------EXPERIMENT ENDED------------------\n\n')

-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=400, NUMBER_OF_SEQUENCE=400, SEQUENCE_LENGTH=10
experiment age: 316 seconds
-------------------EXPERIMENT ENDED------------------


-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=800, NUMBER_OF_SEQUENCE=400, SEQUENCE_LENGTH=10
experiment age: 575 seconds
-------------------EXPERIMENT ENDED------------------


-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=200, NUMBER_OF_SEQUENCE=400, SEQUENCE_LENGTH=20
experiment age: 296 seconds
-------------------EXPERIMENT ENDED------------------


-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=400, NUMBER_OF_SEQUENCE=400, SEQUENCE_LENGTH=20
experiment age: 594 seconds
-------------------EXPERIMENT ENDED------------------


-----------------NEW EXPERIMENT STARTED-----------------
experiment hyperparameters: EPOCHS=800, NUMBER_

KeyboardInterrupt: 